# 07 · Score the July (CP08) population
Rebuild the same 43 Step 02 features with `src/features.build_features` at each customer's `policy_as_of_ts`
(**inclusive**: events ≤ cutoff), then score with the saved Logistic baseline and the four saved T-Learner models.
No optimization here.

## Load data and saved models

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np, pandas as pd, joblib
from src.common import PROC, TABLES, MODELS, load_clean, load_processed, save
from src.features import build_features, FEATURES, CATEGORICAL
from src.modeling import ACTIONS, feature_lists
pd.set_option("display.width", 200)
t = load_clean()
ps = t["policy_scoring_population"]
NUM, CAT = feature_lists()
logit = joblib.load(MODELS / "logistic_propensity_model.joblib")
logit_spec = json.loads((MODELS / "logistic_model_features.json").read_text())
tl = {a: joblib.load(MODELS / f"tlearner_ncm_{a}.joblib") for a in ACTIONS}
print(ps.shape, "| unique customers:", ps.customer_id.nunique(), "| cutoffs:", ps.policy_as_of_ts.unique())

(2776, 3) | unique customers: 2776 | cutoffs: <DatetimeArray>
['2025-06-20 10:00:00']
Length: 1, dtype: datetime64[us]


## Build July features (one row per scoring customer, cutoff = policy_as_of_ts inclusive)

In [2]:
spine = ps.assign(score_id=ps.customer_id)
feats, audit = build_features(spine, t, key="score_id", cut="policy_as_of_ts", inclusive=True)
jf = ps.merge(feats.drop(columns=["customer_id", "policy_as_of_ts"]).rename(columns={"score_id": "customer_id"}), on="customer_id", how="left", validate="1:1")
print(jf.shape)

(2776, 46)


## Leakage and parity tests

In [3]:
checks = {}
lag_h = {k: v / pd.Timedelta(hours=1) for k, v in audit.items()}
checks["max(event_ts - policy_as_of_ts) <= 0 for every family"] = all(v <= 0 for v in lag_h.values())
cut = ps.policy_as_of_ts.max()
after = {"orders": (t["orders"].order_ts > cut).sum(), "returns": (t["returns"].return_ts > cut).sum(),
         "web_sessions": (t["web_sessions"].session_ts > cut).sum(), "email_events": (t["email_events"].event_ts > cut).sum()}
checks["no CP08 outcome rows exist (assignments / orders)"] = not (t["historical_assignments"].campaign_id.eq("CP08_POLICY").any() or t["orders"].campaign_id.eq("CP08_POLICY").any())
# Perturbation: inject events strictly after each cutoff; features must not move
n = len(ps); late = ps.policy_as_of_ts + pd.Timedelta(minutes=1)
tp = {k: v.copy() for k, v in t.items()}
tp["orders"] = pd.concat([tp["orders"], pd.DataFrame({"order_id": [f"FX{i}" for i in range(n)], "customer_id": ps.customer_id.values, "order_ts": late.values,
                          "campaign_id": "CP08_POLICY", "order_source": "campaign_attributed", "order_status": "completed", "offer_redeemed": 1,
                          "offer_type_redeemed": "PCT_20", "merchandise_gross_revenue": 500.0, "discount_amount": 100.0, "shipping_revenue": 0.0,
                          "net_revenue": 400.0, "cogs": 250.0, "shipping_cost": 9.0, "is_completed": True})], ignore_index=True)
tp["order_items"] = pd.concat([tp["order_items"], pd.DataFrame({"order_id": [f"FX{i}" for i in range(n)], "line_number": 1, "product_id": "CLM_001",
                               "quantity": 3, "unit_list_price": 166.67, "unit_cost": 83.33})])
prior_o = ps.merge(t["orders"], on="customer_id").query("order_ts <= policy_as_of_ts").drop_duplicates("customer_id")
tp["returns"] = pd.concat([tp["returns"], pd.DataFrame({"return_id": [f"RX{i}" for i in range(len(prior_o))], "order_id": prior_o.order_id.values,
                           "return_ts": (prior_o.policy_as_of_ts + pd.Timedelta(minutes=1)).values, "refund_amount": 50.0, "return_reason": "Other"})])
tp["web_sessions"] = pd.concat([tp["web_sessions"], pd.DataFrame({"session_id": [f"SX{i}" for i in range(n)], "customer_id": pd.array(ps.customer_id.values, dtype="Int64"),
                                "session_ts": late.values, "device_type": "mobile", "pages_viewed": 12, "added_to_cart": 1, "category_viewed": "Camping",
                                "traffic_source": "email", "is_identified": True})])
tp["email_events"] = pd.concat([tp["email_events"], pd.DataFrame({"email_event_id": [f"EX{i}" for i in range(n)], "assignment_id": "AX", "customer_id": ps.customer_id.values,
                                "campaign_id": "CP08_POLICY", "event_ts": late.values, "event_type": "open"})])
f_pert, _ = build_features(spine, tp, key="score_id", cut="policy_as_of_ts", inclusive=True)
pd.testing.assert_frame_equal(feats[FEATURES], f_pert[FEATURES])
checks["perturbation: events 1 min after cutoff change no feature"] = True
hist = load_processed("historical_feature_outcome")
checks["feature names == historical schema (43)"] = set(FEATURES) == set(NUM + CAT) and len(NUM + CAT) == 43 and set(NUM + CAT) <= set(jf.columns)
checks["logistic spec features present"] = set(logit_spec["numeric_features"] + logit_spec["categorical_features"]) <= set(jf.columns)
checks["numeric/categorical types match history"] = all((pd.api.types.is_numeric_dtype(jf[c]) == pd.api.types.is_numeric_dtype(hist[c])) for c in FEATURES)
checks["every scoring customer present exactly once"] = jf.customer_id.is_unique and set(jf.customer_id) == set(ps.customer_id)
aud = pd.DataFrame({"check": list(checks), "status": ["PASS" if v else "FAIL" for v in checks.values()], "value": ""})
aud = pd.concat([aud, pd.DataFrame({"check": [f"max (event_ts - cutoff) hours: {k}" for k in lag_h] + [f"raw {k} rows after cutoff (excluded)" for k in after],
                                    "status": "INFO", "value": [round(v, 2) for v in lag_h.values()] + list(after.values())})])
aud.to_csv(TABLES / "july_scoring_audit.csv", index=False)
assert all(checks.values()), checks
aud

,check,status,value
0,max(event_ts - policy_as_of_ts) <= 0 for every...,PASS,
1,no CP08 outcome rows exist (assignments / orders),PASS,
2,perturbation: events 1 min after cutoff change...,PASS,
3,feature names == historical schema (43),PASS,
4,logistic spec features present,PASS,
5,numeric/categorical types match history,PASS,
6,every scoring customer present exactly once,PASS,
0,max (event_ts - cutoff) hours: purchase,INFO,-3876.3
1,max (event_ts - cutoff) hours: purchase_refunds,INFO,-3611.95
2,max (event_ts - cutoff) hours: prior_assignments,INFO,-24.0


## Feature summary vs history (missingness highlights)

In [4]:
cmp = pd.DataFrame({"july_missing_rate": jf[FEATURES].isna().mean(), "hist_missing_rate": hist[FEATURES].isna().mean(),
                    "july_mean": jf[NUM].mean(), "hist_mean": hist[NUM].mean()}).round(3)
cmp[(cmp.july_missing_rate - cmp.hist_missing_rate).abs() > 0.15]

,july_missing_rate,hist_missing_rate,july_mean,hist_mean
days_since_last_email_click,1.0,0.830,NaN,51.251
days_since_last_email_open,1.0,0.604,NaN,46.338
historical_click_rate,1.0,0.407,NaN,0.195
historical_open_rate,1.0,0.407,NaN,0.535
prior_offer_redemption_rate,1.0,0.509,NaN,0.117


## Score: Logistic propensity + T-Learner NCM under all four actions

In [5]:
sc = jf[["customer_id", "campaign_id", "policy_as_of_ts"]].copy()
sc["predicted_purchase_probability"] = logit.predict_proba(jf[logit_spec["numeric_features"] + logit_spec["categorical_features"]])[:, 1]
for a in ACTIONS:
    sc[f"predicted_ncm_{a}"] = tl[a].predict(jf[NUM + CAT])
for a in ACTIONS:
    sc[f"incremental_ncm_{a}"] = sc[f"predicted_ncm_{a}"] - sc["predicted_ncm_NO_OFFER"]
pcols = [f"predicted_ncm_{a}" for a in ACTIONS]
sc["best_action_unconstrained"] = sc[pcols].idxmax(axis=1).str.replace("predicted_ncm_", "")
sc["best_incremental_ncm_unconstrained"] = sc[[f"incremental_ncm_{a}" for a in ACTIONS]].max(axis=1)
assert np.isfinite(sc[pcols + ["predicted_purchase_probability"]].values).all() and len(sc) == len(ps)
print("propensity:", sc.predicted_purchase_probability.describe()[["min", "mean", "max"]].round(3).to_dict())
pd.DataFrame({"mean_predicted_ncm": sc[pcols].mean().values, "mean_incremental_ncm": [sc[f"incremental_ncm_{a}"].mean() for a in ACTIONS],
              "share_positive_incremental": [(sc[f"incremental_ncm_{a}"] > 0).mean() for a in ACTIONS],
              "unconstrained_best_share": [(sc.best_action_unconstrained == a).mean() for a in ACTIONS]}, index=ACTIONS).round(3)

propensity: {'min': 0.059, 'mean': 0.13, 'max': 0.423}


,mean_predicted_ncm,mean_incremental_ncm,share_positive_incremental,unconstrained_best_share
NO_OFFER,16.150999,0.000,0.000,0.115
PCT_10,13.292000,-2.858,0.454,0.082
PCT_20,9.701000,-6.449,0.004,0.000
FREE_SHIP,18.334000,2.184,0.851,0.803


## Save outputs

In [6]:
save(jf, "july_features")
sc.to_csv(TABLES / "july_action_ncm_scores.csv", index=False)
print("saved data/processed/july_features.{csv,parquet}", jf.shape, "| outputs/tables/july_action_ncm_scores.csv", sc.shape)

saved data/processed/july_features.{csv,parquet} (2776, 46) | outputs/tables/july_action_ncm_scores.csv (2776, 14)
